# How a Python Script Works

Python is usually described as an **interpreted language**, but that does not mean it skips compilation. The reference CPython implementation compiles source code to bytecode and then executes that bytecode in the Python Virtual Machine (PVM).

## Learning goals

- Understand the stages between `.py` source code and program output
- Explain what parsing, compilation, bytecode, and execution do
- Understand `.pyc` cache files and `__pycache__`
- Distinguish a Python script from an imported module
- Inspect bytecode using standard-library tools

## Execution flow

```mermaid
flowchart TD
    A[Python source: .py] --> B[Tokenizer/Lexer]
    B --> C[Parser]
    C --> D[Abstract Syntax Tree AST]
    D --> E[CPython compiler]
    E --> F[Bytecode and code object]
    F --> G[Python Virtual Machine PVM]
    G --> H[System calls / libraries]
    H --> I[Program output]
```

The stages are:

1. **Read the source** - the interpreter reads the `.py` file.
2. **Tokenize** - the source is split into tokens such as names, operators, strings, and indentation.
3. **Parse** - the parser checks syntax and creates an **Abstract Syntax Tree (AST)**.
4. **Compile** - CPython converts the AST into bytecode instructions and creates a code object.
5. **Execute** - the PVM interprets the bytecode one instruction at a time.
6. **Interact with the system** - library code, built-ins, OS calls, and hardware can perform the requested work.

In [ ]:
price = 100
tax = 18
total = price + (price * tax // 100)

print("Total:", total)

## 1. Source code, tokenization, and parsing

The source code is the text written in a file. Tokenization converts this text into small units. Parsing checks grammar and grouping, and the AST represents the structure of the program.

For example, this expression:

```python
total = price + tax
```

has a structure similar to:

```text
Assignment
`-- target: total
`-- value: Addition
    |-- left: price
    `-- right: tax
```

If there is invalid syntax, Python reports a `SyntaxError` before normal bytecode execution begins.

## 2. The compiler and bytecode

The CPython compiler translates the AST into lower-level **bytecode instructions**. A bytecode instruction is not directly the machine code of your CPU. It is designed for the Python Virtual Machine.

Important points:

- The compiler produces a `code` object, which contains bytecode and metadata.
- Bytecode is an intermediate representation, not the original source text.
- It is often faster to execute than repeatedly parsing the source text.
- CPython bytecode is implementation-specific. It can differ between Python versions and implementations.
- The `compile()` and `dis` modules let us inspect this process.

In [ ]:
import dis

def calculate_total(price, tax):
    return price + (price * tax // 100)

code_object = compile(
    "result = calculate_total(100, 18)",
    filename="<learning-example>",
    mode="exec",
)

print("Compiled type:", type(code_object).__name__)
dis.dis(code_object)

In [ ]:
dis.dis(calculate_total)

## 3. Python Virtual Machine (PVM)

The PVM executes bytecode instructions and manages Python runtime behavior. In CPython, the PVM is an interpreter loop implemented in C.

During execution, the PVM:

- Loads constants and names
- Uses a value stack
- Calls built-in and Python functions
- Allocates objects
- Manages `try`/`except` and loop state
- Calls C extensions and operating-system APIs when needed
- Keeps track of execution frames

CPython converts bytecode into native machine instructions through normal C execution. It is therefore different from directly compiling a Python program into one standalone native executable.

## 4. `.pyc` files and `__pycache__`

CPython caches compiled bytecode in files with a `.pyc` suffix. For an imported module, these files are normally stored under `__pycache__` and include Python and implementation details in their filename.

Example:

```text
__pycache__/helper.cpython-311.pyc
```

The cache is a performance optimization:

- Import Python source and CPython finds or creates a valid cache file.
- CPython compares source metadata with the cache header.
- If source is unchanged, it can load bytecode instead of compiling source again.
- If source changed, it recompiles and updates the cache where permitted.
- The cache does not remove the PVM or eliminate runtime interpretation.

Running a top-level `.py` script directly usually does not create a `.pyc` cache for that script itself. Imported modules may be cached.

In [ ]:
import os
import py_compile
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    source = os.path.join(temp_dir, "example.py")
    with open(source, "w", encoding="utf-8") as file:
        file.write("def greet(name):\n    return f'Hello, {name}'\n")

    bytecode_path = py_compile.compile(source, doraise=True)
    print("Source:", source)
    print("Bytecode:", bytecode_path)
    print("Cache directory:", os.path.dirname(bytecode_path))

## Script versus module

The same `.py` file can be used in two ways:

| Usage | Example | `__name__` | Typical behavior |
|---|---|---|---|
| Main script | `python app.py` | `"__main__"` | Executed as the entry point |
| Imported module | `import app` | Module name, such `"app"` | Definitions execute, but module code is reused |
| Module execution | `python -m app` | `"__main__"` | Module is located and run as the entry point |

The conventional entry-point guard prevents application code from running during import:

```python
def main():
    print("Application started")

if __name__ == "__main__":
    main()
```

## CPython and other implementations

Most learners use **CPython**, the standard implementation. It uses a bytecode interpreter written in C and is the implementation distributed from python.org.

Other implementations may differ:

- **PyPy** uses a just-in-time (JIT) compiler and may execute bytecode or generate native code dynamically.
- **Jython** targets the Java Virtual Machine.
- **IronPython** targets the .NET runtime.

Therefore, "Python executes bytecode" is a useful CPython-oriented description, while the language specification does not require every implementation to use identical bytecode.

## Function-call execution behind the scenes

```mermaid
sequenceDiagram
    participant User
    participant PVM
    participant Function
    participant Library
    User->>PVM: Execute main bytecode
    PVM->>PVM: Resolve function name
    PVM->>Function: Create frame and pass arguments
    Function->>PVM: Execute function bytecode
    PVM->>Library: Call built-in or C extension if needed
    Library-->>PVM: Return result
    PVM-->>Function: Return result
    Function-->>PVM: Remove frame
    PVM-->>User: Send output or raise exception
```

## Summary

| Term | Meaning |
|---|---|
| Source code | Human-readable Python text in `.py` files |
| Tokenizer | Splits source text into lexical tokens |
| Parser | Checks syntax and builds an AST |
| AST | Structured tree representing program syntax |
| Compiler | Converts the AST into bytecode and code objects |
| Bytecode | Low-level instructions for the CPython PVM |
| PVM | Runtime loop that executes bytecode |
| `.pyc` | Cached bytecode for faster future imports |
| `__pycache__` | Directory commonly containing `.pyc` files |

### One-line mental model

**Python source is parsed into an AST, compiled to bytecode, and executed instruction by instruction by the PVM.**